# Audit and publish BabyMoE checkpoints

Run the dry-run audit first. Publication creates a public repository, maps the true final checkpoint to `main`, and uploads only genuine `chck_{N}M` revisions.

In [ ]:
GITHUB_REPO = "https://github.com/Amos-Luna/BabyMoE_Research.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/BabyMoE"
RUN_NAME = "moe_learned_seed42"
HF_REPO_ID = "Amos-Luna/BabyMoE-2026-Multilingual"
VERIFY_REMOTE_MODELS = True

In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive")
repo_dir = Path("/content/BabyMoE_Research")
if (repo_dir / ".git").exists():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, GITHUB_REPO, str(repo_dir)], check=True)
os.chdir(repo_dir)
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
subprocess.run(["bash", "scripts/colab_bootstrap.sh", "--drive-root", DRIVE_ROOT], check=True)
RUN_DIR = f"{DRIVE_ROOT}/artifacts/runs/{RUN_NAME}"

In [ ]:
subprocess.run([
    "python", "scripts/upload_to_hf.py",
    "--run-dir", RUN_DIR,
    "--repo-id", HF_REPO_ID,
    "--dry-run",
], check=True, env=os.environ.copy())

In [ ]:
command = [
    "python", "scripts/upload_to_hf.py",
    "--run-dir", RUN_DIR,
    "--repo-id", HF_REPO_ID,
]
if VERIFY_REMOTE_MODELS:
    command.append("--verify-remote-models")
subprocess.run(command, check=True, env=os.environ.copy())